# Filtering and Retraining

## Imports, paths, and baseline-only configuration

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Union, Sequence, List, Tuple, Optional, Dict
import random
import json
import gc
import time

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import segmentation_models_pytorch as smp
import optuna

import matplotlib.pyplot as plt

RUN_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

img_path_full = Path("../inputs/map/map_test.tif")
lbl_path_full = Path("../inputs/label/label_test.tif")

baseline_split_dir = Path("../outputs/splits/unet_resnet34_p256_b2048")
train_csv = baseline_split_dir / "train_patches.csv"
val_csv = baseline_split_dir / "val_patches.csv"
test_csv = baseline_split_dir / "test_patches.csv"

experiment_name = "unet_filtering_comparison_10seeds"
out_dir = Path(f"../outputs/proposed/{experiment_name}")
out_dir.mkdir(parents=True, exist_ok=True)
tables_dir = out_dir / "tables"; tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir = out_dir / "figures"; figures_dir.mkdir(parents=True, exist_ok=True)
checkpoints_dir = out_dir / "checkpoints"; checkpoints_dir.mkdir(parents=True, exist_ok=True)
optuna_dir = out_dir / "optuna"; optuna_dir.mkdir(parents=True, exist_ok=True)

# Persistent handoff used by the two Decision Tree notebooks.
SHARED_OPTUNA_DIR = Path("../outputs/proposed/unet_baseline_optuna_shared")
SHARED_OPTUNA_DIR.mkdir(parents=True, exist_ok=True)
SHARED_OPTUNA_PARAMS_PATH = Path("../outputs/proposed/unet_baseline_optuna_shared/baseline_optuna_best_params.json")

num_classes = 3
label_values = [0, 1, 2]
CLASS_NAMES = {0: "Non-deforestation", 1: "Deforestation", 2: "Cloud"}
nodata_value = -9999
ignore_index = -9999
MASK_REMOVE_VALUE = 1
MASK_RETAIN_VALUE = 0
MASK_NODATA_VALUE = 255
in_channels = 6
bands = ["R", "G", "B", "NIR", "SWIR1", "SWIR2"]
patch_size = 256
stride_train = 256
block_size = 2048
ratios = (0.70, 0.15, 0.15)
FIXED_ENCODER = "resnet34"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 0
PIN_MEMORY = device.type == "cuda"
FINAL_MAX_EPOCHS = 50
FIXED_PATIENCE = 8
N_TRIALS_OPTUNA = 10
OPTUNA_MAX_EPOCHS = 5
OPTUNA_PATIENCE = 5
OPTUNA_SEED = 42
OPTUNA_DB_PATH = optuna_dir / "optuna_baseline.db"
MEAN_STD_MAX_PATCHES = 400
CLASS_FREQ_MAX_PATCHES = 400

for path in [img_path_full, lbl_path_full, train_csv, val_csv, test_csv]:
    assert path.exists(), f"Missing required file: {path}"

print("Device:", device)
print("Output directory:", out_dir)
print("Shared Optuna parameter file:", SHARED_OPTUNA_PARAMS_PATH)

## ============================================================

In [ ]:
# ============================================================
# ============================================================

@dataclass(frozen=True)
class PatchSpec:
    row_off: int
    col_off: int
    height: int
    width: int


def load_patch_specs(csv_path: Union[str, Path]) -> List[PatchSpec]:
    df = pd.read_csv(csv_path)

    required = {"row_off", "col_off", "height", "width"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(f"Missing columns in {csv_path}: {missing}")

    return [
        PatchSpec(
            row_off=int(r.row_off),
            col_off=int(r.col_off),
            height=int(r.height),
            width=int(r.width),
        )
        for r in df.itertuples(index=False)
    ]


def export_patch_specs(
    patches: Sequence[PatchSpec],
    csv_path: Union[str, Path],
) -> pd.DataFrame:

    csv_path = Path(csv_path)

    df = pd.DataFrame([
        {
            "row_off": int(p.row_off),
            "col_off": int(p.col_off),
            "height": int(p.height),
            "width": int(p.width),
        }
        for p in patches
    ])

    df.to_csv(csv_path, index=False)
    return df


def patch_key(p: PatchSpec) -> Tuple[int, int, int, int]:
    return (
        int(p.row_off),
        int(p.col_off),
        int(p.height),
        int(p.width),
    )


def set_all_seeds(seed_value: int) -> None:
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)

    # Keep cudnn benchmark enabled for speed; exact GPU determinism is not guaranteed.
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


## Load the common patch splits

In [ ]:
all_train_patches = load_patch_specs(train_csv)
val_patches = load_patch_specs(val_csv)
test_patches = load_patch_specs(test_csv)

train_keys = {patch_key(p) for p in all_train_patches}
val_keys = {patch_key(p) for p in val_patches}
test_keys = {patch_key(p) for p in test_patches}
assert len(train_keys & val_keys) == 0
assert len(train_keys & test_keys) == 0
assert len(val_keys & test_keys) == 0

# Baseline uses no training mask, so no mask-alignment check is required.

export_patch_specs(all_train_patches, out_dir / "train_patches.csv")
export_patch_specs(val_patches, out_dir / "val_patches.csv")
export_patch_specs(test_patches, out_dir / "test_patches.csv")

split_summary = pd.DataFrame([
    {"split": "train_shared", "n_patches": len(all_train_patches)},
    {"split": "validation_shared", "n_patches": len(val_patches)},
    {"split": "test_shared", "n_patches": len(test_patches)},
])
split_summary.to_csv(out_dir / "split_summary.csv", index=False)
display(split_summary)

## ============================================================

In [ ]:
# ============================================================
# ============================================================

def estimate_mean_std(
    img_path: Union[str, Path],
    patch_list: Sequence[PatchSpec],
    nodata_value: int = -9999,
    max_patches: int = 400,
    seed: int = 42,
) -> Tuple[np.ndarray, np.ndarray]:

    if len(patch_list) == 0:
        raise ValueError("Cannot estimate mean/std from an empty patch list.")

    rng = np.random.default_rng(seed)
    n = min(max_patches, len(patch_list))
    sample_indices = rng.choice(len(patch_list), size=n, replace=False)

    sums = None
    sumsqs = None
    counts = None

    with rasterio.open(img_path) as img:
        for idx in sample_indices:
            p = patch_list[int(idx)]
            w = Window(p.col_off, p.row_off, p.width, p.height)

            x = img.read(window=w).astype(np.float32)
            valid = x != nodata_value

            if sums is None:
                C = x.shape[0]
                sums = np.zeros(C, dtype=np.float64)
                sumsqs = np.zeros(C, dtype=np.float64)
                counts = np.zeros(C, dtype=np.int64)

            for c in range(x.shape[0]):
                vals = x[c][valid[c]]
                counts[c] += vals.size

                if vals.size > 0:
                    vals64 = vals.astype(np.float64)
                    sums[c] += vals64.sum()
                    sumsqs[c] += (vals64 ** 2).sum()

    mean = sums / np.maximum(counts, 1)
    var = (sumsqs / np.maximum(counts, 1)) - mean ** 2
    std = np.sqrt(np.maximum(var, 1e-12))

    return mean.astype(np.float32), std.astype(np.float32)


def estimate_class_freqs(
    lbl_path: Union[str, Path],
    patch_list: Sequence[PatchSpec],
    num_classes: int,
    ignore_index: int = -9999,
    max_patches: int = 400,
    seed: int = 42,
    training_mask_path: Optional[Union[str, Path]] = None,
    mask_remove_value: int = 1,
    mask_nodata_value: int = 255,
) -> np.ndarray:

    if len(patch_list) == 0:
        raise ValueError("Cannot estimate class frequencies from an empty patch list.")

    rng = np.random.default_rng(seed)
    n = min(max_patches, len(patch_list))
    sample_indices = rng.choice(len(patch_list), size=n, replace=False)
    counts = np.zeros(num_classes, dtype=np.int64)

    mask_src = rasterio.open(training_mask_path) if training_mask_path is not None else None

    try:
        with rasterio.open(lbl_path) as lbl:
            for idx in sample_indices:
                p = patch_list[int(idx)]
                w = Window(p.col_off, p.row_off, p.width, p.height)
                y = lbl.read(1, window=w)
                valid = y != ignore_index

                if mask_src is not None:
                    mask = mask_src.read(1, window=w)
                    valid &= mask != mask_remove_value
                    valid &= mask != mask_nodata_value

                yv = y[valid]
                if yv.size == 0:
                    continue

                for cls in range(num_classes):
                    counts[cls] += np.sum(yv == cls)
    finally:
        if mask_src is not None:
            mask_src.close()

    total = counts.sum()
    if total == 0:
        raise ValueError("No valid retained training pixels were found.")

    return counts / total


def make_loss_weights(
    freqs: np.ndarray,
    beta: float,
    device: torch.device,
) -> Tuple[np.ndarray, torch.Tensor]:

    weights = (1.0 / freqs) ** beta
    weights = weights / np.mean(weights)

    weights_t = torch.tensor(
        weights,
        dtype=torch.float32,
        device=device,
    )

    return weights.astype(np.float32), weights_t


## Dataset with optional pixel-level training mask

In [ ]:
#
# Every sample is returned as a fixed patch_size × patch_size tensor.
# Edge patches in the split CSV may have smaller recorded dimensions
# (for example 6 × 6). Rasterio boundless reading pads those samples:
#   imagery -> nodata_value
#   labels  -> ignore_index
#   masks   -> MASK_NODATA_VALUE
# This prevents small edge patches from reaching the ResNet encoder.
# ============================================================

class RasterPatchDataset(Dataset):
    def __init__(
        self,
        img_path: Union[str, Path],
        lbl_path: Union[str, Path],
        patches: Sequence[PatchSpec],
        num_classes: int = 3,
        nodata_value: int = -9999,
        ignore_index: int = -9999,
        mean: Optional[np.ndarray] = None,
        std: Optional[np.ndarray] = None,
        augment: bool = False,
        aug_prob: float = 0.0,
        noise_std: float = 0.0,
        seed: int = 42,
        training_mask_path: Optional[Union[str, Path]] = None,
        mask_remove_value: int = 1,
        mask_nodata_value: int = 255,
        output_patch_size: int = patch_size,
    ):
        self.img_path = str(img_path)
        self.lbl_path = str(lbl_path)
        self.patches = list(patches)
        self.num_classes = num_classes
        self.nodata_value = nodata_value
        self.ignore_index = ignore_index
        self.mean = None if mean is None else mean.reshape(-1, 1, 1)
        self.std = None if std is None else std.reshape(-1, 1, 1)
        self.augment = bool(augment)
        self.aug_prob = float(aug_prob)
        self.noise_std = float(noise_std)
        self.rng = np.random.default_rng(seed)
        self.training_mask_path = (
            None if training_mask_path is None else str(training_mask_path)
        )
        self.mask_remove_value = int(mask_remove_value)
        self.mask_nodata_value = int(mask_nodata_value)
        self.output_patch_size = int(output_patch_size)

        if self.output_patch_size < 32:
            raise ValueError(
                "output_patch_size must be at least 32 pixels for the ResNet encoder."
            )

    def __len__(self):
        return len(self.patches)

    def _augment(self, x, y):
        if self.rng.random() > self.aug_prob:
            return x, y
        if self.rng.random() < 0.5:
            x = x[:, :, ::-1].copy()
            y = y[:, ::-1].copy()
        if self.rng.random() < 0.5:
            x = x[:, ::-1, :].copy()
            y = y[::-1, :].copy()
        k = int(self.rng.integers(0, 4))
        if k:
            x = np.rot90(x, k=k, axes=(1, 2)).copy()
            y = np.rot90(y, k=k, axes=(0, 1)).copy()
        if self.noise_std > 0:
            noise = self.rng.normal(
                0.0,
                self.noise_std,
                size=x.shape,
            ).astype(np.float32)
            valid = x != self.nodata_value
            x = np.where(valid, x + noise, x).astype(np.float32)
        return x, y

    def __getitem__(self, idx: int):
        p = self.patches[idx]

        # Always request the configured fixed spatial size. For patches at
        # raster boundaries, boundless=True pads outside-raster pixels.
        w = Window(
            int(p.col_off),
            int(p.row_off),
            self.output_patch_size,
            self.output_patch_size,
        )

        with rasterio.open(self.img_path) as img, rasterio.open(self.lbl_path) as lbl:
            x = img.read(
                window=w,
                boundless=True,
                fill_value=self.nodata_value,
            ).astype(np.float32)
            y = lbl.read(
                1,
                window=w,
                boundless=True,
                fill_value=self.ignore_index,
            ).astype(np.int64)

        expected_x_shape = (
            in_channels,
            self.output_patch_size,
            self.output_patch_size,
        )
        expected_y_shape = (
            self.output_patch_size,
            self.output_patch_size,
        )

        if x.shape != expected_x_shape:
            raise RuntimeError(
                f"Unexpected image patch shape at index {idx}: "
                f"{x.shape}; expected {expected_x_shape}."
            )
        if y.shape != expected_y_shape:
            raise RuntimeError(
                f"Unexpected label patch shape at index {idx}: "
                f"{y.shape}; expected {expected_y_shape}."
            )

        # Apply masks only to training dataset instances that receive a mask path.
        if self.training_mask_path is not None:
            with rasterio.open(self.training_mask_path) as mask_src:
                mask = mask_src.read(
                    1,
                    window=w,
                    boundless=True,
                    fill_value=self.mask_nodata_value,
                )

            if mask.shape != expected_y_shape:
                raise RuntimeError(
                    f"Unexpected mask patch shape at index {idx}: "
                    f"{mask.shape}; expected {expected_y_shape}."
                )

            remove_or_nodata = (
                (mask == self.mask_remove_value)
                | (mask == self.mask_nodata_value)
            )
            y[remove_or_nodata] = self.ignore_index

        if self.augment:
            x, y = self._augment(x, y)

        if self.mean is not None and self.std is not None:
            valid = x != self.nodata_value
            x_norm = (x - self.mean) / self.std
            x = np.where(valid, x_norm, 0.0).astype(np.float32)

        return torch.from_numpy(x), torch.from_numpy(y)

## Verify fixed dataset tensor shapes before Optuna starts

In [ ]:
small_patch_count = sum(
    (p.height < patch_size) or (p.width < patch_size)
    for p in (all_train_patches + val_patches + test_patches)
)

print(
    f"Patch records smaller than {patch_size} × {patch_size}: "
    f"{small_patch_count}"
)

_shape_check_mean = np.zeros(in_channels, dtype=np.float32)
_shape_check_std = np.ones(in_channels, dtype=np.float32)

for split_name, split_patches in [
    ("train", all_train_patches),
    ("validation", val_patches),
    ("test", test_patches),
]:
    if not split_patches:
        raise ValueError(f"The {split_name} split is empty.")

    check_indices = sorted(set([0, len(split_patches) - 1]))
    check_ds = RasterPatchDataset(
        img_path=img_path_full,
        lbl_path=lbl_path_full,
        patches=split_patches,
        num_classes=num_classes,
        nodata_value=nodata_value,
        ignore_index=ignore_index,
        mean=_shape_check_mean,
        std=_shape_check_std,
        augment=False,
        output_patch_size=patch_size,
    )

    for idx in check_indices:
        x_check, y_check = check_ds[idx]
        assert tuple(x_check.shape) == (in_channels, patch_size, patch_size)
        assert tuple(y_check.shape) == (patch_size, patch_size)

    print(
        f"{split_name}: verified fixed tensor shape "
        f"({in_channels}, {patch_size}, {patch_size})"
    )

print("Dataset shape checks passed. Optuna can now run safely.")

## ============================================================

In [ ]:
# ============================================================
# ============================================================

def build_model():
    return smp.Unet(
        encoder_name=FIXED_ENCODER,
        encoder_weights=None,
        in_channels=in_channels,
        classes=num_classes,
    )


@torch.no_grad()
def mean_iou(
    pred_logits: torch.Tensor,
    y_true: torch.Tensor,
    num_classes: int = 3,
    ignore_index: int = -9999,
) -> float:

    preds = torch.argmax(pred_logits, dim=1)

    ious = []

    for cls in range(num_classes):
        pred_mask = preds == cls
        true_mask = y_true == cls

        valid = y_true != ignore_index

        pred_mask = pred_mask & valid
        true_mask = true_mask & valid

        intersection = (pred_mask & true_mask).sum().item()
        union = (pred_mask | true_mask).sum().item()

        if union == 0:
            continue

        ious.append(intersection / union)

    if len(ious) == 0:
        return float("nan")

    return float(np.mean(ious))


def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    print_every: int = 25,
    verbose: bool = True,
) -> Tuple[float, float]:

    model.train()

    running_loss = 0.0
    running_iou = []

    if verbose:
        print(
            f"    Training epoch with {len(loader)} batches...",
            flush=True,
        )

    for batch_idx, (x, y) in enumerate(loader, start=1):

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)

        running_iou.append(
            mean_iou(
                logits,
                y,
                num_classes=num_classes,
                ignore_index=ignore_index,
            )
        )

        if verbose and (
            batch_idx == 1
            or batch_idx % print_every == 0
            or batch_idx == len(loader)
        ):
            print(
                f"      Batch {batch_idx:04d}/{len(loader):04d} | "
                f"batch_loss={loss.item():.4f}",
                flush=True,
            )

    epoch_loss = running_loss / len(loader.dataset)
    epoch_iou = float(np.nanmean(running_iou))

    return epoch_loss, epoch_iou


@torch.no_grad()
def update_confusion_matrix(
    cm: torch.Tensor,
    preds: torch.Tensor,
    targets: torch.Tensor,
    num_classes: int,
    ignore_index: int,
):
    valid = targets != ignore_index

    preds = preds[valid]
    targets = targets[valid]

    if preds.numel() == 0:
        return cm

    idx = targets * num_classes + preds

    binc = torch.bincount(
        idx,
        minlength=num_classes * num_classes,
    ).reshape(num_classes, num_classes)

    cm += binc.to(cm.device)

    return cm


@torch.no_grad()
def metrics_from_confusion(cm: torch.Tensor) -> dict:
    cm = cm.to(torch.float64)

    tp = torch.diag(cm)
    fp = cm.sum(dim=0) - tp
    fn = cm.sum(dim=1) - tp

    eps = 1e-12

    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    iou = tp / (tp + fp + fn + eps)

    present = cm.sum(dim=1) > 0

    if present.any():
        macro_precision = precision[present].mean().item()
        macro_recall = recall[present].mean().item()
        macro_f1 = f1[present].mean().item()
        macro_iou = iou[present].mean().item()
    else:
        macro_precision = float("nan")
        macro_recall = float("nan")
        macro_f1 = float("nan")
        macro_iou = float("nan")

    accuracy = (tp.sum() / (cm.sum() + eps)).item()

    return {
        "per_class": {
            "precision": precision.cpu().numpy(),
            "recall": recall.cpu().numpy(),
            "f1": f1.cpu().numpy(),
            "iou": iou.cpu().numpy(),
            "support": cm.sum(dim=1).cpu().numpy(),
        },
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "macro_iou": macro_iou,
        "accuracy": accuracy,
        "confusion_matrix": cm.cpu().numpy(),
    }


@torch.no_grad()
def evaluate_loader(
    model,
    loader,
    num_classes: int,
    ignore_index: int,
    device,
) -> dict:

    model.eval()

    cm = torch.zeros((num_classes, num_classes), dtype=torch.int64)

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        preds = torch.argmax(logits, dim=1).to(torch.int64)

        cm = update_confusion_matrix(
            cm,
            preds.cpu(),
            y.cpu(),
            num_classes=num_classes,
            ignore_index=ignore_index,
        )

    return metrics_from_confusion(cm)


def metrics_to_frames(
    metrics: dict,
    split_name: str,
    experiment: str,
    seed_value: int,
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    pc = metrics["per_class"]

    rows = []

    for c in range(num_classes):
        rows.append({
            "experiment": experiment,
            "seed": int(seed_value),
            "split": split_name,
            "class": c,
            "class_name": CLASS_NAMES[c],
            "precision": float(pc["precision"][c]),
            "recall": float(pc["recall"][c]),
            "f1": float(pc["f1"][c]),
            "iou": float(pc["iou"][c]),
            "support_pixels": float(pc["support"][c]),
            "accuracy": float(metrics["accuracy"]),
        })

    macro = {
        "experiment": experiment,
        "seed": int(seed_value),
        "split": split_name,
        "class": "macro",
        "class_name": "Macro",
        "precision": float(metrics["macro_precision"]),
        "recall": float(metrics["macro_recall"]),
        "f1": float(metrics["macro_f1"]),
        "iou": float(metrics["macro_iou"]),
        "accuracy": float(metrics["accuracy"]),
    }

    return pd.DataFrame(rows), pd.DataFrame([macro])


## Optuna tuning for one training regime

In [ ]:
def run_optuna_for_experiment(
    experiment_label: str,
    train_patches_for_tuning: Sequence[PatchSpec],
    val_patches_for_tuning: Sequence[PatchSpec],
    training_mask_path: Optional[Union[str, Path]] = None,
    n_trials: int = N_TRIALS_OPTUNA,
    optuna_max_epochs: int = OPTUNA_MAX_EPOCHS,
    patience: int = OPTUNA_PATIENCE,
    seed_value: int = OPTUNA_SEED,
) -> Dict:

    allowed = ["baseline"]
    assert experiment_label in allowed
    assert n_trials <= 10, "Maximum Optuna trials must not exceed 10."
    assert optuna_max_epochs == 5, "Optuna must use exactly 5 maximum epochs."
    assert patience == 5, "Optuna patience must be fixed at 5."

    set_all_seeds(seed_value)
    tuning_dir = optuna_dir / experiment_label
    tuning_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 72)
    print(f"Optuna tuning: {experiment_label}")
    print(f"Training mask: {training_mask_path}")
    print(f"Trials: {n_trials} | maximum epochs: {optuna_max_epochs} | patience: {patience}")
    print("=" * 72)

    train_mean, train_std = estimate_mean_std(
        img_path=img_path_full,
        patch_list=train_patches_for_tuning,
        nodata_value=nodata_value,
        max_patches=MEAN_STD_MAX_PATCHES,
        seed=seed_value,
    )

    freqs = estimate_class_freqs(
        lbl_path=lbl_path_full,
        patch_list=train_patches_for_tuning,
        num_classes=num_classes,
        ignore_index=ignore_index,
        max_patches=CLASS_FREQ_MAX_PATCHES,
        seed=seed_value,
        training_mask_path=training_mask_path,
        mask_remove_value=MASK_REMOVE_VALUE,
        mask_nodata_value=MASK_NODATA_VALUE,
    )

    np.save(tuning_dir / "tuning_train_mean.npy", train_mean)
    np.save(tuning_dir / "tuning_train_std.npy", train_std)

    def objective(trial: optuna.Trial) -> float:
        trial_seed = seed_value + trial.number
        set_all_seeds(trial_seed)

        # Only these six hyperparameters are compared by Optuna.
        params = {
            "lr": trial.suggest_float("lr", 1e-5, 3e-3, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
            "batch_size": trial.suggest_categorical("batch_size", [4, 8, 16]),
            "class_weight_beta": trial.suggest_float("class_weight_beta", 0.0, 1.0),
            "aug_prob": trial.suggest_float("aug_prob", 0.0, 0.8),
            "noise_std": trial.suggest_float("noise_std", 0.0, 0.10),
        }

        _, weights_t = make_loss_weights(freqs, params["class_weight_beta"], device)

        train_ds = RasterPatchDataset(
            img_path_full, lbl_path_full, train_patches_for_tuning,
            num_classes=num_classes, nodata_value=nodata_value,
            ignore_index=ignore_index, mean=train_mean, std=train_std,
            augment=True, aug_prob=params["aug_prob"],
            noise_std=params["noise_std"], seed=trial_seed,
            training_mask_path=training_mask_path,
            mask_remove_value=MASK_REMOVE_VALUE,
            mask_nodata_value=MASK_NODATA_VALUE,
        )
        val_ds = RasterPatchDataset(
            img_path_full, lbl_path_full, val_patches_for_tuning,
            num_classes=num_classes, nodata_value=nodata_value,
            ignore_index=ignore_index, mean=train_mean, std=train_std,
            augment=False, seed=trial_seed,
        )

        train_loader = DataLoader(train_ds, batch_size=params["batch_size"], shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
        val_loader = DataLoader(val_ds, batch_size=params["batch_size"], shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

        model = build_model().to(device)
        criterion = nn.CrossEntropyLoss(ignore_index=ignore_index, weight=weights_t)
        optimizer = optim.AdamW(model.parameters(), lr=params["lr"],
                                weight_decay=params["weight_decay"])

        best_val = -1.0
        no_improve = 0
        try:
            for epoch in range(1, optuna_max_epochs + 1):
                train_one_epoch(model, train_loader, optimizer, criterion, device,
                                print_every=999999, verbose=False)
                val_metrics = evaluate_loader(model, val_loader, num_classes,
                                              ignore_index, device)
                val_iou = float(val_metrics["macro_iou"])
                trial.report(val_iou, step=epoch)
                if trial.should_prune():
                    raise optuna.TrialPruned()
                if np.isfinite(val_iou) and val_iou > best_val + 1e-4:
                    best_val = val_iou; no_improve = 0
                else:
                    no_improve += 1
                if no_improve >= patience:
                    break
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                if torch.cuda.is_available(): torch.cuda.empty_cache()
                raise optuna.TrialPruned()
            raise
        finally:
            del model, train_ds, val_ds, train_loader, val_loader
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        return float(best_val)

    study = optuna.create_study(
        direction="maximize",
        storage=f"sqlite:///{OPTUNA_DB_PATH.resolve()}",
        study_name=f"{experiment_name}_{experiment_label}",
        load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=seed_value),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=3, interval_steps=1),
    )
    study.optimize(objective, n_trials=n_trials)

    df_trials = study.trials_dataframe().sort_values("value", ascending=False)
    # Explicitly omit any legacy max-epoch or patience columns from comparisons.
    drop_cols = [c for c in df_trials.columns if "max_epochs" in c or "patience" in c]
    df_trials = df_trials.drop(columns=drop_cols, errors="ignore")
    trials_path = tuning_dir / "optuna_trials.csv"
    df_trials.to_csv(trials_path, index=False)

    best = study.best_params
    best_params_for_final = {
        "lr": float(best["lr"]),
        "weight_decay": float(best["weight_decay"]),
        "batch_size": int(best["batch_size"]),
        "class_weight_beta": float(best["class_weight_beta"]),
        "aug_prob": float(best["aug_prob"]),
        "noise_std": float(best["noise_std"]),
        "optuna_best_value": float(study.best_value),
    }

    params_path = tuning_dir / "best_params_for_final_run.json"
    with open(params_path, "w") as f:
        json.dump(best_params_for_final, f, indent=2)

    print("Best Optuna value:", study.best_value)
    print(json.dumps(best_params_for_final, indent=2))
    return best_params_for_final

## experiment_label: str,

In [ ]:
def load_or_run_optuna_for_experiment(
    experiment_label: str,
    train_patches_for_tuning: Sequence[PatchSpec],
    val_patches_for_tuning: Sequence[PatchSpec],
    training_mask_path: Optional[Union[str, Path]] = None,
    n_trials: int = N_TRIALS_OPTUNA,
    optuna_max_epochs: int = OPTUNA_MAX_EPOCHS,
    patience: int = OPTUNA_PATIENCE,
    seed_value: int = OPTUNA_SEED,
    force_rerun: bool = False,
) -> Dict:
    tuning_dir = optuna_dir / experiment_label
    params_path = tuning_dir / "best_params_for_final_run.json"

    if params_path.exists() and not force_rerun:
        print(f"Loading existing Optuna parameters for {experiment_label}: {params_path}")
        with open(params_path, "r") as f:
            return json.load(f)

    return run_optuna_for_experiment(
        experiment_label=experiment_label,
        train_patches_for_tuning=train_patches_for_tuning,
        val_patches_for_tuning=val_patches_for_tuning,
        training_mask_path=training_mask_path,
        n_trials=n_trials,
        optuna_max_epochs=optuna_max_epochs,
        patience=patience,
        seed_value=seed_value,
    )


## ============================================================

In [ ]:
# ============================================================
# ============================================================

def run_single_seed_experiment(
    experiment_label: str,
    train_patches_for_run: Sequence[PatchSpec],
    val_patches_for_run: Sequence[PatchSpec],
    test_patches_for_run: Sequence[PatchSpec],
    seed_value: int,
    best_params_for_run: Dict,
    training_mask_path: Optional[Union[str, Path]] = None,
    save_checkpoint: bool = True,
) -> Dict[str, pd.DataFrame]:

    assert experiment_label in ["baseline"]

    set_all_seeds(seed_value)

    run_dir = out_dir / experiment_label / f"seed_{seed_value}"
    run_dir.mkdir(parents=True, exist_ok=True)

    with open(run_dir / "best_params_used.json", "w") as f:
        json.dump(best_params_for_run, f, indent=2)

    print("=" * 72)
    print(f"Running experiment: {experiment_label} | seed={seed_value}")
    print(f"Train patches: {len(train_patches_for_run)}")
    print(f"Val patches:   {len(val_patches_for_run)}")
    print(f"Test patches:  {len(test_patches_for_run)}")
    print("=" * 72)

    # --------------------------------------------------------
    # Train-only normalisation and class weights for this regime
    # --------------------------------------------------------

    train_mean, train_std = estimate_mean_std(
        img_path=img_path_full,
        patch_list=train_patches_for_run,
        nodata_value=nodata_value,
        max_patches=MEAN_STD_MAX_PATCHES,
        seed=seed_value,
    )

    freqs = estimate_class_freqs(
        lbl_path=lbl_path_full,
        patch_list=train_patches_for_run,
        num_classes=num_classes,
        ignore_index=ignore_index,
        max_patches=CLASS_FREQ_MAX_PATCHES,
        seed=seed_value,
        training_mask_path=training_mask_path,
        mask_remove_value=MASK_REMOVE_VALUE,
        mask_nodata_value=MASK_NODATA_VALUE,
    )

    weights, weights_t = make_loss_weights(
        freqs=freqs,
        beta=best_params_for_run["class_weight_beta"],
        device=device,
    )

    np.save(run_dir / "train_mean.npy", train_mean)
    np.save(run_dir / "train_std.npy", train_std)

    pd.DataFrame({
        "class": list(range(num_classes)),
        "class_name": [CLASS_NAMES[c] for c in range(num_classes)],
        "frequency_estimate": freqs,
        "loss_weight": weights,
    }).to_csv(run_dir / "class_frequencies_and_weights.csv", index=False)

    # --------------------------------------------------------
    # Datasets and loaders
    # --------------------------------------------------------

    train_ds = RasterPatchDataset(
        img_path=img_path_full,
        lbl_path=lbl_path_full,
        patches=train_patches_for_run,
        num_classes=num_classes,
        nodata_value=nodata_value,
        ignore_index=ignore_index,
        mean=train_mean,
        std=train_std,
        augment=True,
        aug_prob=best_params_for_run["aug_prob"],
        noise_std=best_params_for_run["noise_std"],
        seed=seed_value,
        training_mask_path=training_mask_path,
        mask_remove_value=MASK_REMOVE_VALUE,
        mask_nodata_value=MASK_NODATA_VALUE,
    )

    val_ds = RasterPatchDataset(
        img_path=img_path_full,
        lbl_path=lbl_path_full,
        patches=val_patches_for_run,
        num_classes=num_classes,
        nodata_value=nodata_value,
        ignore_index=ignore_index,
        mean=train_mean,
        std=train_std,
        augment=False,
        seed=seed_value,
    )

    test_ds = RasterPatchDataset(
        img_path=img_path_full,
        lbl_path=lbl_path_full,
        patches=test_patches_for_run,
        num_classes=num_classes,
        nodata_value=nodata_value,
        ignore_index=ignore_index,
        mean=train_mean,
        std=train_std,
        augment=False,
        seed=seed_value,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=best_params_for_run["batch_size"],
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=best_params_for_run["batch_size"],
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=best_params_for_run["batch_size"],
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    # --------------------------------------------------------
    # Model and optimisation
    # --------------------------------------------------------

    model = build_model().to(device)

    criterion = nn.CrossEntropyLoss(
        ignore_index=ignore_index,
        weight=weights_t,
    )

    optimizer = optim.AdamW(
        model.parameters(),
        lr=best_params_for_run["lr"],
        weight_decay=best_params_for_run["weight_decay"],
    )

    best_ckpt = run_dir / "final_model_best.pt"
    history_path = run_dir / "train_history.csv"

    best_val_macro_iou = -1.0
    no_improve = 0
    history = []

    start_time = time.time()

    for epoch in range(1, FINAL_MAX_EPOCHS + 1):
        train_loss, train_batch_miou = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        print_every=999999,
        verbose=False,
    )

        val_metrics = evaluate_loader(
            model=model,
            loader=val_loader,
            num_classes=num_classes,
            ignore_index=ignore_index,
            device=device,
        )

        row = {
            "experiment": experiment_label,
            "seed": int(seed_value),
            "epoch": epoch,
            "train_loss": float(train_loss),
            "train_batch_miou": float(train_batch_miou),
            "val_macro_precision": float(val_metrics["macro_precision"]),
            "val_macro_recall": float(val_metrics["macro_recall"]),
            "val_macro_f1": float(val_metrics["macro_f1"]),
            "val_macro_iou": float(val_metrics["macro_iou"]),
            "val_accuracy": float(val_metrics["accuracy"]),
        }

        history.append(row)

        print(
            f"Epoch {epoch:03d} | "
            f"loss={row['train_loss']:.4f} | "
            f"train_batch_mIoU={row['train_batch_miou']:.4f} | "
            f"val_mIoU={row['val_macro_iou']:.4f} | "
            f"val_acc={row['val_accuracy']:.4f}",
            flush=True,
        )

        if row["val_macro_iou"] > best_val_macro_iou + 1e-4:
            best_val_macro_iou = row["val_macro_iou"]
            no_improve = 0

            if save_checkpoint:
                torch.save(
                    {
                        "model_state_dict": model.state_dict(),
                        "best_params": best_params_for_run,
                        "train_mean": train_mean,
                        "train_std": train_std,
                        "experiment": experiment_label,
                        "seed": int(seed_value),
                        "baseline_split_dir": str(baseline_split_dir),
                        "training_mask_path": None if training_mask_path is None else str(training_mask_path),
                    },
                    best_ckpt,
                )
        else:
            no_improve += 1

            if no_improve >= FIXED_PATIENCE:
                print("Early stopping triggered.")
                break

    history_df = pd.DataFrame(history)
    history_df.to_csv(history_path, index=False)

    # --------------------------------------------------------
    # Reload best checkpoint and evaluate val/test
    # --------------------------------------------------------

    if save_checkpoint and best_ckpt.exists():
        ckpt = torch.load(best_ckpt, map_location=device)
        model = build_model().to(device)
        model.load_state_dict(ckpt["model_state_dict"])
        model.eval()

    val_metrics = evaluate_loader(
        model=model,
        loader=val_loader,
        num_classes=num_classes,
        ignore_index=ignore_index,
        device=device,
    )

    test_metrics = evaluate_loader(
        model=model,
        loader=test_loader,
        num_classes=num_classes,
        ignore_index=ignore_index,
        device=device,
    )

    df_val_pc, df_val_macro = metrics_to_frames(
        val_metrics,
        "val",
        experiment_label,
        seed_value,
    )

    df_test_pc, df_test_macro = metrics_to_frames(
        test_metrics,
        "test",
        experiment_label,
        seed_value,
    )

    per_class_df = pd.concat([df_val_pc, df_test_pc], ignore_index=True)
    macro_df = pd.concat([df_val_macro, df_test_macro], ignore_index=True)

    per_class_df.to_csv(run_dir / "final_per_class_metrics.csv", index=False)
    macro_df.to_csv(run_dir / "final_macro_metrics.csv", index=False)

    pd.DataFrame(val_metrics["confusion_matrix"]).to_csv(
        run_dir / "confusion_matrix_val.csv",
        index=False,
    )

    pd.DataFrame(test_metrics["confusion_matrix"]).to_csv(
        run_dir / "confusion_matrix_test.csv",
        index=False,
    )

    runtime_seconds = time.time() - start_time

    run_info = pd.DataFrame([{
        "experiment": experiment_label,
        "seed": int(seed_value),
        "n_train_patches": len(train_patches_for_run),
        "n_val_patches": len(val_patches_for_run),
        "n_test_patches": len(test_patches_for_run),
        "best_val_macro_iou": float(best_val_macro_iou),
        "n_epochs_ran": int(len(history_df)),
        "runtime_seconds": float(runtime_seconds),
        "checkpoint_path": str(best_ckpt) if save_checkpoint else "",
        "lr": float(best_params_for_run["lr"]),
        "weight_decay": float(best_params_for_run["weight_decay"]),
        "batch_size": int(best_params_for_run["batch_size"]),
        "class_weight_beta": float(best_params_for_run["class_weight_beta"]),
        "aug_prob": float(best_params_for_run["aug_prob"]),
        "noise_std": float(best_params_for_run["noise_std"]),
        "fixed_patience": int(FIXED_PATIENCE),
        "training_mask_path": None if training_mask_path is None else str(training_mask_path),
    }])

    run_info.to_csv(run_dir / "run_info.csv", index=False)

    print(f"Finished {experiment_label} | seed={seed_value}")
    print(f"Runtime minutes: {runtime_seconds / 60:.2f}")
    print("Final macro metrics:")
    display(macro_df)

    # --------------------------------------------------------
    # Clean up GPU memory between runs
    # --------------------------------------------------------

    del model, train_ds, val_ds, test_ds, train_loader, val_loader, test_loader
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "history": history_df,
        "macro": macro_df,
        "per_class": per_class_df,
        "run_info": run_info,
    }


## Three retraining conditions

In [ ]:
# These are the two masks exported by Notebook 04.
# Mask convention inherited from the thesis notebooks:
#   1   = remove pixel from training loss
#   0   = retain pixel
#   255 = mask nodata
ORIGINAL_DT_MASK = Path(
    "../notebooks/QGIS_calibration_error/outputs/decision_tree_only/"
    "decision_tree_original_label_error_mask.tif"
)
DT_85_MASK = Path(
    "../notebooks/QGIS_calibration_error/outputs/decision_tree_only/"
    "decision_tree_label_error_probability_85pct_mask.tif"
)

EXPERIMENTS = {
    "baseline": None,
    "original_dt_mask": ORIGINAL_DT_MASK,
    "probability_85_mask": DT_85_MASK,
}

for name, mask_path in EXPERIMENTS.items():
    if mask_path is not None:
        assert mask_path.exists(), f"Missing mask for {name}: {mask_path}"

print("Experiments:")
for name, mask_path in EXPERIMENTS.items():
    print(f"  {name:24s} -> {mask_path}")

## ============================================================

In [ ]:
# ============================================================
# Tune the baseline once, then reuse identical parameters
# for all three 10-seed experiments
# ============================================================

baseline_best_params = load_or_run_optuna_for_experiment(
    experiment_label="baseline",
    train_patches_for_tuning=all_train_patches,
    val_patches_for_tuning=val_patches,
    training_mask_path=None,
)

shared_best_params = dict(baseline_best_params)
with open(SHARED_OPTUNA_PARAMS_PATH, "w") as f:
    json.dump(shared_best_params, f, indent=2)

print("Shared baseline Optuna parameters:")
print(json.dumps(shared_best_params, indent=2))

all_results = {}

for experiment_label, mask_path in EXPERIMENTS.items():
    print("\n" + "=" * 72)
    print("Running:", experiment_label)
    print("=" * 72)

    all_results[experiment_label] = [
        run_single_seed_experiment(
            experiment_label=experiment_label,
            train_patches_for_run=all_train_patches,
            val_patches_for_run=val_patches,
            test_patches_for_run=test_patches,
            seed_value=seed_value,
            best_params_for_run=shared_best_params,
            training_mask_path=mask_path,
        )
        for seed_value in RUN_SEEDS
    ]


## Combine, export, and compare all 10-seed results

In [ ]:
macro_frames = []
per_class_frames = []
history_frames = []
run_info_frames = []

for experiment_label, results in all_results.items():
    macro_frames.extend(r["macro"] for r in results)
    per_class_frames.extend(r["per_class"] for r in results)
    history_frames.extend(r["history"] for r in results)
    run_info_frames.extend(r["run_info"] for r in results)

df_macro = pd.concat(macro_frames, ignore_index=True)
df_per_class = pd.concat(per_class_frames, ignore_index=True)
df_history = pd.concat(history_frames, ignore_index=True)
df_run_info = pd.concat(run_info_frames, ignore_index=True)

df_macro.to_csv(tables_dir / "all_experiments_macro_metrics_10seeds.csv", index=False)
df_per_class.to_csv(tables_dir / "all_experiments_per_class_metrics_10seeds.csv", index=False)
df_history.to_csv(tables_dir / "all_experiments_training_history_10seeds.csv", index=False)
df_run_info.to_csv(tables_dir / "all_experiments_run_info_10seeds.csv", index=False)

metric_cols = ["precision", "recall", "f1", "iou", "accuracy"]
comparison = (
    df_macro
    .groupby(["experiment", "split"])[metric_cols]
    .agg(["mean", "std"])
    .reset_index()
)
comparison.to_csv(tables_dir / "filtering_retraining_10seed_summary.csv", index=False)

display(comparison)